# Multimodal Fusion Training Strategies

Train different fusion approaches to improve accuracy from 60% (image-only) to 80%+

**Estimated Time per Model:** 45-60 minutes  
**Total Time:** 3-4 hours for all strategies

---

## Setup: Import and Check Environment

In [ ]:
import os
import sys
from pathlib import Path

# Ensure we're in the correct directory
os.chdir(Path(__file__).parent if '__file__' in dir() else Path.cwd())
print(f"Working directory: {os.getcwd()}")

# Check if training data exists
assert Path('train.csv').exists(), "train.csv not found!"
assert Path('train_images').exists(), "train_images/ folder not found!"
print("✓ Training data found")

# Check if test data exists
assert Path('test_release/index_text_test.csv').exists(), "test_release/index_text_test.csv not found!"
assert Path('test_release/test_images').exists(), "test_release/test_images/ folder not found!"
print("✓ Test data found")

## Helper Functions

In [ ]:
def update_config(modality, fusion_strategy, freeze_encoders=True, hidden_dim=512, lr=1e-4):
    """Update config.py with specified parameters"""
    with open('config.py', 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Update modality
    content = content.replace("USE_MODALITY = 'image'", f"USE_MODALITY = '{modality}'")
    content = content.replace("USE_MODALITY = 'text'", f"USE_MODALITY = '{modality}'")
    content = content.replace("USE_MODALITY = 'both'", f"USE_MODALITY = '{modality}'")
    
    # Update fusion strategy
    content = content.replace("FUSION_STRATEGY = 'early'", f"FUSION_STRATEGY = '{fusion_strategy}'")
    content = content.replace("FUSION_STRATEGY = 'late_voting'", f"FUSION_STRATEGY = '{fusion_strategy}'")
    content = content.replace("FUSION_STRATEGY = 'late_bagging'", f"FUSION_STRATEGY = '{fusion_strategy}'")
    content = content.replace("FUSION_STRATEGY = 'hybrid'", f"FUSION_STRATEGY = '{fusion_strategy}'")
    
    # Update encoder freezing
    freeze_val = str(freeze_encoders)
    content = content.replace("FREEZE_TEXT_ENCODER = True", f"FREEZE_TEXT_ENCODER = {freeze_val}")
    content = content.replace("FREEZE_TEXT_ENCODER = False", f"FREEZE_TEXT_ENCODER = {freeze_val}")
    content = content.replace("FREEZE_IMAGE_ENCODER = True", f"FREEZE_IMAGE_ENCODER = {freeze_val}")
    content = content.replace("FREEZE_IMAGE_ENCODER = False", f"FREEZE_IMAGE_ENCODER = {freeze_val}")
    
    # Update hidden dimension
    import re
    content = re.sub(r'HIDDEN_DIM = \d+', f'HIDDEN_DIM = {hidden_dim}', content)
    
    # Update learning rate
    content = re.sub(r'LEARNING_RATE = [0-9.e-]+', f'LEARNING_RATE = {lr}', content)
    
    with open('config.py', 'w', encoding='utf-8') as f:
        f.write(content)
    
    print(f"✓ Config updated:")
    print(f"  - Modality: {modality}")
    print(f"  - Fusion: {fusion_strategy}")
    print(f"  - Freeze encoders: {freeze_encoders}")
    print(f"  - Hidden dim: {hidden_dim}")
    print(f"  - Learning rate: {lr}")

def list_trained_models():
    """List all trained models in Output/"""
    output_dir = Path('Output')
    if not output_dir.exists():
        print("No Output/ directory found")
        return
    
    models = sorted(output_dir.glob('*'))
    print(f"\n{'Model Directory':<50} {'Folds'}")
    print("-" * 60)
    for model_dir in models:
        if model_dir.is_dir():
            fold_count = len(list(model_dir.glob('best_model_fold_*.pt')))
            print(f"{model_dir.name:<50} {fold_count}")

print("✓ Helper functions loaded")

---

## Strategy 1: Early Fusion (Simple Concatenation)

**Expected Accuracy:** 70-78%  
**Description:** Concatenates image+text embeddings early, then applies fusion layers  
**Best for:** Fast training, good baseline multimodal performance

In [ ]:
# Configure for Early Fusion
update_config(
    modality='both',
    fusion_strategy='early',
    freeze_encoders=True,
    hidden_dim=512,
    lr=1e-4
)

In [ ]:
# Train Early Fusion Model
!python train.py

---

## Strategy 2: Late Voting (Majority Vote)

**Expected Accuracy:** 72-80%  
**Description:** Separate image/text classifiers, majority vote on predictions  
**Best for:** Robust predictions, handles modality imbalance

In [ ]:
# Configure for Late Voting
update_config(
    modality='both',
    fusion_strategy='late_voting',
    freeze_encoders=True,
    hidden_dim=512,
    lr=1e-4
)

In [ ]:
# Train Late Voting Model
!python train.py

---

## Strategy 3: Late Bagging (Probability Averaging)

**Expected Accuracy:** 73-81%  
**Description:** Separate image/text classifiers, average probability scores  
**Best for:** Smoother predictions, confidence calibration

In [ ]:
# Configure for Late Bagging
update_config(
    modality='both',
    fusion_strategy='late_bagging',
    freeze_encoders=True,
    hidden_dim=512,
    lr=1e-4
)

In [ ]:
# Train Late Bagging Model
!python train.py

---

## Strategy 4: Hybrid Fusion (Frozen Encoders)

**Expected Accuracy:** 75-82%  
**Description:** Early concat + late voting combined with frozen encoders  
**Best for:** Best balance of speed and accuracy

In [ ]:
# Configure for Hybrid Fusion (Frozen)
update_config(
    modality='both',
    fusion_strategy='hybrid',
    freeze_encoders=True,
    hidden_dim=512,
    lr=1e-4
)

In [ ]:
# Train Hybrid Fusion (Frozen) Model
!python train.py

---

## Strategy 5: Hybrid Fusion (Fine-tuned Encoders)

**Expected Accuracy:** 78-85%  
**Description:** Early concat + late voting with encoder fine-tuning  
**Best for:** Maximum accuracy (slower training)

In [ ]:
# Configure for Hybrid Fusion (Fine-tuned)
update_config(
    modality='both',
    fusion_strategy='hybrid',
    freeze_encoders=False,
    hidden_dim=512,
    lr=2e-5  # Lower LR for fine-tuning
)

In [ ]:
# Train Hybrid Fusion (Fine-tuned) Model
!python train.py

---

## Check Trained Models

In [ ]:
# List all trained models
list_trained_models()

---

## Generate Predictions for All Models

In [ ]:
# List available models to test
!python generate_submission.py --list-models

In [ ]:
# Generate predictions for all models
from pathlib import Path
import subprocess

submissions_dir = Path('submissions')
submissions_dir.mkdir(exist_ok=True)

output_dir = Path('Output')
for model_dir in sorted(output_dir.glob('*')):
    if model_dir.is_dir():
        output_file = submissions_dir / f"{model_dir.name}.csv"
        print(f"\n{'='*60}")
        print(f"Testing: {model_dir.name}")
        print(f"{'='*60}")
        
        result = subprocess.run(
            ['python', 'generate_submission.py', '--model_dir', str(model_dir), '--output', str(output_file)],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            print(f"✓ SUCCESS: {output_file}")
        else:
            print(f"✗ ERROR: {result.stderr}")

print(f"\n✓ All predictions saved to: {submissions_dir}/")

---

## Create Ensemble (Majority Vote Across All Models)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load all submissions
submissions_dir = Path('submissions')
submission_files = sorted([f for f in submissions_dir.glob('*.csv') if 'ENSEMBLE' not in f.stem])

if not submission_files:
    print("No submission files found!")
else:
    print(f"Loading {len(submission_files)} submissions...\n")
    
    # Load all predictions
    all_predictions = []
    for f in submission_files:
        df = pd.read_csv(f)
        all_predictions.append(df['label'].values)
        hate_pct = 100 * df['label'].sum() / len(df)
        print(f"{f.stem:<50} Hate: {hate_pct:5.1f}%")
    
    # Create ensemble by majority voting
    predictions_array = np.array(all_predictions)
    ensemble_predictions = (predictions_array.sum(axis=0) > len(submission_files) / 2).astype(int)
    
    # Save ensemble
    ensemble_df = pd.DataFrame({
        'index': pd.read_csv(submission_files[0])['index'],
        'label': ensemble_predictions
    })
    ensemble_file = submissions_dir / 'ENSEMBLE.csv'
    ensemble_df.to_csv(ensemble_file, index=False)
    
    ensemble_hate_pct = 100 * ensemble_predictions.sum() / len(ensemble_predictions)
    print(f"\n{'='*60}")
    print(f"{'ENSEMBLE (Majority Vote)':<50} Hate: {ensemble_hate_pct:5.1f}%")
    print(f"{'='*60}")
    print(f"\n✓ Saved to: {ensemble_file}")
    print(f"\n🎯 RECOMMENDATION: Submit ENSEMBLE.csv first (typically highest accuracy)")

---

## Summary: Next Steps

1. **Submit to Competition:**
   - Priority 1: `submissions/ENSEMBLE.csv` (Expected: 80-87%)
   - Priority 2: `submissions/both_hybrid_*_fine-tuned*.csv` (Expected: 78-85%)
   - Priority 3: `submissions/both_hybrid_*.csv` (Expected: 75-82%)

2. **If accuracy < 80%:**
   - Try different hyperparameters (HIDDEN_DIM, learning rate)
   - Experiment with data augmentation
   - Add more training epochs

3. **Interactive Model Selection:**
   ```python
   !python generate_submission.py --interactive
   ```